In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
file = r"C:\Users\Desire Luminsa\Downloads\enrollments (1).csv"

In [3]:
df = pd.read_csv(file)

In [4]:
df['Service Type'] = df['Service Type'].astype(str)
df = df[df['Service Type'].str.contains('ART')].copy()

In [5]:
a = df.shape[0]

In [6]:
df = df[['MR - First name', 'MR - Surname', 'MR - Sex' ,'HIV/ART-Next Appointment date', 'Last updated on','ART: Art Number','HIV-ART Regimen - No. of days dispensed','Service Type']]

In [7]:
df[['HIV/ART-Next Appointment date', 'Last updated on']] = (df[['HIV/ART-Next Appointment date', 'Last updated on']]
                                                            .apply(lambda col: pd.to_datetime(col,format='mixed',dayfirst=True).dt.date))

In [8]:
dfart = df[df['ART: Art Number'].isnull()].copy()
dfartn = df[df['ART: Art Number'].notnull()].copy()

In [9]:
dfartn['ART'] = dfartn['ART: Art Number'].astype(str).str.replace('[^0-9]', '', regex=True)

In [10]:
dfartn['ART'] = dfartn['ART'].fillna(0)

In [11]:
dfartn['ART'] = pd.to_numeric(dfartn['ART'], errors = 'coerce')

In [12]:
dfartna = dfartn[dfartn['ART']<1].copy()

In [13]:
if dfartna.shape[0]>0:
    dfart = pd.concat([dfartna, dfart])
    dfartn = dfartn[dfartn['ART']>0].copy()
    

In [14]:
dfartn['ART'] = dfartn['ART: Art Number'].astype(str).str.replace('[^0-9]', '', regex=True)

In [15]:
dfart['ART_STATUS'] = 'NO ART NUMBER'

In [16]:
no_art = dfart.shape[0]

In [17]:
dfdup = dfartn[dfartn['ART'].duplicated()].copy()

In [18]:
dfdup['DUP STATUS'] = 'DUPLICATED IN E-REGISTER'

In [19]:
dup_ereg = dfdup.shape[0]

In [20]:
dfnodup = dfartn[~dfartn['ART'].duplicated()].copy()

In [21]:
if dfdup.shape[0]>0:
    dfartn = pd.concat([dfdup, dfnodup])
else:
    dfartn = dfnodup

In [22]:
if dfart.shape[0]>0:
    df = pd.concat([dfart, dfartn])
else:
    df = dfartn

In [23]:
df['HIV-ART Regimen - No. of days dispensed'] = pd.to_numeric(df['HIV-ART Regimen - No. of days dispensed'], errors = 'coerce')

In [24]:
dfnopills = df[df['HIV-ART Regimen - No. of days dispensed'].isnull()].copy()#NO PILLS 
dfpills = df[df['HIV-ART Regimen - No. of days dispensed'].notnull()].copy() #HAS PILLS

In [25]:
dfnoday = dfnopills[((dfnopills['HIV-ART Regimen - No. of days dispensed'].isnull()) & (dfnopills['HIV/ART-Next Appointment date'].isnull()))].copy()

In [26]:
dfday = dfnopills[((dfnopills['HIV-ART Regimen - No. of days dispensed'].isnull()) & (dfnopills['HIV/ART-Next Appointment date'].notnull()))].copy()

In [27]:
dfday = dfday.drop(columns=['HIV-ART Regimen - No. of days dispensed'])

In [28]:
dfday[['HIV/ART-Next Appointment date', 'Last updated on']] = (dfday[['HIV/ART-Next Appointment date', 'Last updated on']]
                                                            .apply(lambda col: pd.to_datetime(col,format='mixed',dayfirst=True)))

In [29]:
dfday['HIV-ART Regimen - No. of days dispensed'] = ( dfday['HIV/ART-Next Appointment date'] - dfday['Last updated on']).dt.days

In [30]:
dfnoday['DAYS_STATUS'] = 'MISSING DAYS DISPENSED'

In [31]:
dfnoday = dfnoday.drop(columns =['HIV/ART-Next Appointment date','HIV-ART Regimen - No. of days dispensed'])

In [32]:
#dfa = pd.concat([dfday, dfnoday])
dfs = [dfx for dfx in [dfday, dfnoday] if not df.empty]

dfa = pd.concat(dfs, ignore_index=True)

In [33]:
dfnodate = dfpills[dfpills['HIV/ART-Next Appointment date'].isnull()].copy()

In [34]:
dfdate = dfpills[dfpills['HIV/ART-Next Appointment date'].notna()].copy()

In [35]:
dfnodate = dfnodate.drop(columns =['HIV/ART-Next Appointment date'])

In [36]:
dfnodate['Last updated on'] = pd.to_datetime(dfnodate['Last updated on'],format='mixed',dayfirst=True)

In [37]:
dfnodate['HIV-ART Regimen - No. of days dispensed'] = pd.to_numeric(dfnodate['HIV-ART Regimen - No. of days dispensed'], errors='coerce')

In [38]:
dfnodate['days'] = pd.to_timedelta(dfnodate['HIV-ART Regimen - No. of days dispensed'],unit='D')

In [39]:
dfnodate['HIV/ART-Next Appointment date'] = dfnodate['Last updated on'] + dfnodate['days']

In [40]:
#dfb = pd.concat([dfdate, dfnodate])
dfs = [dfx for dfx in [dfdate, dfnodate] if not df.empty]

dfb = pd.concat(dfs, ignore_index=True)

In [41]:
#df = pd.concat([dfa, dfb])
dfs = [dfx for dfx in [dfa, dfb] if not df.empty]

df = pd.concat(dfs, ignore_index=True)

In [42]:
def pillcheck(a):
    if 0 <= a <30:
        return 'FEW DAYS DISPENSED, CHECK'
    elif 29 < a <186:
        'OK'
    elif a > 185:
        'MANY DAYS DISPENSED, CHECK'
    else:
        pass

In [43]:
df['HIV-ART Regimen - No. of days dispensed'] = pd.to_numeric(df['HIV-ART Regimen - No. of days dispensed'], errors='coerce').copy()
df['DAYS ERROR']  = df['HIV-ART Regimen - No. of days dispensed'].apply(pillcheck)

In [44]:
dfmany = df[df['DAYS ERROR']=='MANY DAYS DISPENSED'].copy()
dfew = df[df['DAYS ERROR']== 'FEW DAYS DISPENSED, CHECK'].copy()

In [45]:
df = df[~df['DAYS ERROR'].isin(['MANY DAYS DISPENSED, CHECK','FEW DAYS DISPENSED, CHECK'])].copy()

In [46]:
checkd = {'NO ART NOs': dfart.shape[0],
          'DUPLICATED IN E-REG': dfdup.shape[0],
          'NO DAYS DISPENSED' : dfnoday.shape[0],
          'FEW DAYS DISPENSED' : dfew.shape[0],
          'TOO MANY DAYS DIS[ENSED': dfmany.shape[0]
}

In [47]:
for key,value in checkd.items():
    if value>0:
        print(f'{key}: {value}')

NO ART NOs: 53
DUPLICATED IN E-REG: 25
NO DAYS DISPENSED: 99
FEW DAYS DISPENSED: 7


In [48]:
b =df.shape[0] + dfmany.shape[0] + dfew.shape[0]

In [49]:
a

530

In [50]:
df.shape

(523, 14)

In [51]:
if a-b !=0:
    print('warning')
else:
    print('passed')

passed


In [52]:
df = df[['MR - First name', 'MR - Surname', 'MR - Sex' ,'HIV/ART-Next Appointment date', 'Last updated on','ART: Art Number','HIV-ART Regimen - No. of days dispensed']]

In [53]:
df = df.rename(columns = {'HIV/ART-Next Appointment date':'Return Visit Date', 'Last updated on':'Last Encounter Date',
                          'HIV-ART Regimen - No. of days dispensed': 'Days Dispensed'})

In [54]:
df[['Return Visit Date','Last Encounter Date']] = df[['Return Visit Date','Last Encounter Date']].apply(lambda col: pd.to_datetime(col,format='mixed',dayfirst=True))

In [55]:
df['Rday'] = df['Return Visit Date'].dt.day
df['Rmonth'] = df['Return Visit Date'].dt.month
df['Ryear'] = df['Return Visit Date'].dt.year

In [56]:
df['Lday'] = df['Last Encounter Date'].dt.day
df['Lmonth'] = df['Last Encounter Date'].dt.month
df['Lyear'] = df['Last Encounter Date'].dt.year

In [58]:
df['ART'] = df['ART: Art Number'].astype(str).str.replace('[^0-9]', "", regex= True)

In [60]:
file2 = r"C:\Users\Desire Luminsa\Desktop\PROJECT CENTCOM\PROGRAMS\EMR\BATCH_REFERENCE\MATEETE.csv"

In [61]:
df2 = pd.read_csv(file2)

In [62]:
df2['ART'] = df2['Art'].astype(str).str.replace('[^0-9]', "", regex= True)

In [63]:
df2['ARVS'] = df2['ARVS'].astype(str).str.replace('/', '-')

In [64]:
df2['ART'] = pd.to_numeric(df2['ART'], errors = 'coerce')

In [65]:
df['ART'] = pd.to_numeric(df['ART'], errors = 'coerce')

In [66]:
df2 = df2[df2['ART'].notna()].copy()
df = df[df['ART'].notna()].copy()

In [67]:
df['ART'] = pd.to_numeric(df['ART'], errors = 'coerce')

In [68]:
df = df.drop_duplicates(subset= ['ART'], keep='first') ####DUPS WON'T PASS ANYWAY, REMOVE LATER

In [69]:
df['ART'] = pd.to_numeric(df['ART'], errors = 'coerce')
df2['ART'] = pd.to_numeric(df2['ART'], errors = 'coerce')

In [70]:
df = pd.merge(df, df2, on ='ART', how = 'left')

In [71]:
dfdupe =  df[df['ART'].duplicated()].copy()
df =  df[~df['ART'].duplicated()].copy()

In [72]:
dfdup['REASON_REJECTED'] = 'DUPLICATED IN EMR, UPDATE ONE BY ONE'

In [73]:
dfnoart = df[df['Art'].isnull()].copy()
df = df[df['Art'].notnull()].copy()

In [74]:
dfnoart['REASON_REJECTED'] = 'NOT IN EMR, MAY BE TX_NEWS/VISITORS'

In [75]:
file2 = r"C:\Users\Desire Luminsa\Desktop\CV"

In [76]:
out = os.path.join(file2, 'ALLS.csv')

In [79]:
df = df[['Art', 'Days Dispensed', 'Rday','Rmonth', 'Ryear', 'Lday', 'Lmonth', 'Lyear', 'ART', 'Art', 'ARVS']].copy()